# Overlay multiple .dat files and show channel + total events

Reads multiple `.dat` files, plots energy vs count for each on the same axes, and shows in the legend the channel name and total events (raw total × 5000×360).

In [ ]:
import os
import matplotlib.pyplot as plt

# --- Configuration ---
base_dir = 'out_copy'  # or 'out'
# List of .dat filenames to overlay (change to your files)
dat_filenames = [
    'stpi_ibd_eos_events_smeared_unweighted.dat',
    'stpi_nue_O16_2_d2O_events_smeared_unweighted.dat',
    'stpi_nuebar_e_eos_events_smeared.dat',
]

FACTOR = 5000 * 360  # total events = raw_total * FACTOR

def channel_from_path(path):
    """Derive channel label from filename (strip path, .dat, and optional _events* suffix)."""
    name = os.path.splitext(os.path.basename(path))[0]
    for suffix in ('_events_smeared_unweighted', '_events_smeared', '_events_unweighted', '_events'):
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name

def get_total_from_file(path):
    """Read file and return the total (from 'Total:' line)."""
    with open(path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    if not lines:
        return 0.0
    last = lines[-1]
    if last.lower().startswith('total:'):
        for p in last.split()[1:]:
            try:
                return float(p)
            except ValueError:
                continue
    # fallback: last data line, second column
    for p in reversed(last.split()):
        try:
            return float(p)
        except ValueError:
            continue
    return 0.0

def read_xy(path):
    """Return (x_vals, y_vals) for plottable data lines (skip separator and Total)."""
    x_vals, y_vals = [], []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('-') or line.lower().startswith('total'):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                x_vals.append(float(parts[0]))
                y_vals.append(float(parts[1]))
            except ValueError:
                continue
    return x_vals, y_vals

In [ ]:
# Resolve paths (try cwd first, then repo root)
def resolve_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if os.path.exists(p):
        return p
    p = os.path.join(os.path.dirname(os.getcwd()), base_dir, filename)
    return p if os.path.exists(p) else os.path.join(base_dir, filename)

paths = [resolve_path(base_dir, f) for f in dat_filenames]
# Filter to existing files
paths = [p for p in paths if os.path.exists(p)]
if not paths:
    raise FileNotFoundError(f'None of the files found. base_dir={base_dir!r}, cwd={os.getcwd()!r}')

In [ ]:
# Plot all files on the same axes; legend = channel + total events (× 5000×360)
fig, ax = plt.subplots(figsize=(9, 5))
for path in paths:
    x_vals, y_vals = read_xy(path)
    total = get_total_from_file(path)
    total_events = total * FACTOR
    channel = channel_from_path(path)
    label = f'{channel}  —  {total_events:,.0f} events'
    ax.plot(x_vals, y_vals, marker='o', linestyle='-', markersize=2, label=label)

ax.set_xlabel('Energy (MeV)')
ax.set_ylabel('Counts')
ax.set_title('Event spectra (total = raw × 5000×360)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()